# Version 5 : Ensemble de Modèles de Survie

**Stratégie d'ensemble** :
- 🌲 **RandomSurvivalForest** (40%) - Robuste, capture interactions
- 📈 **GradientBoostingSurvivalAnalysis** (40%) - Capture non-linéarités
- 📊 **CoxPHSurvivalAnalysis** (20%) - Modèle linéaire baseline

**Configuration** :
- 90 features optimales (de V3.1)
- Hyperparamètres optimaux par modèle
- Moyenne pondérée des prédictions

**Objectif** : C-index > 0.76

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time

# Survival models
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported")

## 2. Configuration (From V4)

In [ ]:
# Best 90 features (from V3.1)
BEST_FEATURES = [
    'BM_BLAST', 'HB', 'PLT', 'vaf_sum', 'mutation_count_total', 'effect_non_synonymous_codon',
    'gene_RUNX1_count', 'WBC', 'cyto_total_anomalies', 'gene_RUNX1_present', 'ANC', 'cyto_normal',
    'MONOCYTES', 'vaf_mean', 'gene_TP53_count', 'cyto_complex', 'gene_TP53_present', 'cyto_loss_count',
    'effect_frameshift_variant', 'gene_NRAS_present', 'gene_NRAS_count', 'CENTER_PV', 'effect_PTD',
    'gene_ASXL1_count', 'cyto_chr7_affected', 'gene_TET2_count', 'gene_SF3B1_present', 'vaf_max',
    'gene_DNMT3A_present', 'gene_U2AF1_present', 'gene_SF3B1_count', 'CENTER_RMCN', 'cyto_chr18_affected',
    'cyto_gain_count', 'gene_U2AF1_count', 'cyto_chr1_affected', 'cyto_chrX_affected', 'cyto_other_count',
    'cyto_transloc_count', 'cyto_chr3_affected', 'gene_NF1_count', 'cyto_chr7_abnormal', 'gene_EZH2_present',
    'cyto_chr6_affected', 'CENTER_TUD', 'gene_SRSF2_count', 'gene_DNMT3A_count', 'gene_PHF6_count',
    'gene_TET2_present', 'CENTER_ROM', 'effect_inframe_codon_gain', 'gene_BCOR_count', 'CENTER_GESMD',
    'gene_CUX1_present', 'cyto_chr4_affected', 'gene_ZRSR2_count', 'gene_ZRSR2_present', 'CENTER_DUS',
    'cyto_chrY_affected', 'CENTER_CGM', 'cyto_del5q', 'cyto_chr15_affected', 'cyto_chr14_affected',
    'gene_NF1_present', 'CENTER_DUTH', 'CENTER_HIAE', 'effect_complex_change_in_transcript',
    'effect_initiator_codon_change', 'cyto_chr3_abnormal', 'cyto_chr19_affected', 'cyto_chr10_affected',
    'CENTER_UMG', 'CENTER_REL', 'CENTER_IHBT', 'CENTER_HMS', 'CENTER_MSK', 'effect_3_prime_UTR_variant',
    'effect_2KB_upstream_variant', 'effect_ITD', 'effect_stop_lost', 'cyto_inv_count',
    'effect_stop_retained_variant', 'effect_inframe_variant', 'effect_synonymous_codon', 'CENTER_VU',
    'CENTER_UOXF', 'CENTER_UOB', 'gene_KRAS_present', 'gene_KRAS_count', 'cyto_has_inv'
]

# Ensemble weights
ENSEMBLE_WEIGHTS = {
    'rsf': 0.40,
    'gbm': 0.40,
    'cox': 0.20
}

print(f"✓ Configuration loaded: {len(BEST_FEATURES)} features")
print(f"✓ Ensemble weights: RSF={ENSEMBLE_WEIGHTS['rsf']}, GBM={ENSEMBLE_WEIGHTS['gbm']}, Cox={ENSEMBLE_WEIGHTS['cox']}")

## 3. Data Loading & Feature Engineering

Réutilisation du code de V4

In [ ]:
DATA_PATH = r"C:\Users\guill\Desktop\Data Challenge QRT\Data-Challenge-Prediction-de-Survie"

clinical_train = pd.read_csv(f"{DATA_PATH}\\X_train\\clinical_train.csv")
target_train = pd.read_csv(f"{DATA_PATH}\\target_train.csv")
clinical_test = pd.read_csv(f"{DATA_PATH}\\X_test\\clinical_test.csv")
molecular_train = pd.read_csv(f"{DATA_PATH}\\X_train\\molecular_train.csv")
molecular_test = pd.read_csv(f"{DATA_PATH}\\X_test\\molecular_test.csv")

print(f"✓ Data loaded: {clinical_train.shape[0]} train patients")

In [ ]:
# Feature engineering functions (from V4)
def create_cytogenetic_features(clinical_df):
    cyto_features = pd.DataFrame(index=clinical_df['ID'])
    cyto_col = clinical_df.set_index('ID')['CYTOGENETICS'].fillna('')
    cyto_features['cyto_del_count'] = cyto_col.str.count(r'del\(')
    cyto_features['cyto_has_del'] = (cyto_features['cyto_del_count'] > 0).astype(int)
    cyto_features['cyto_transloc_count'] = cyto_col.str.count(r't\(')
    cyto_features['cyto_has_transloc'] = (cyto_features['cyto_transloc_count'] > 0).astype(int)
    cyto_features['cyto_inv_count'] = cyto_col.str.count(r'inv\(')
    cyto_features['cyto_has_inv'] = (cyto_features['cyto_inv_count'] > 0).astype(int)
    cyto_features['cyto_gain_count'] = cyto_col.str.count(r'\+')
    cyto_features['cyto_has_gain'] = (cyto_features['cyto_gain_count'] > 0).astype(int)
    cyto_features['cyto_loss_count'] = cyto_col.str.count(r'-[0-9XY]')
    cyto_features['cyto_has_loss'] = (cyto_features['cyto_loss_count'] > 0).astype(int)
    cyto_features['cyto_other_count'] = cyto_col.str.count(r'add\(|ins\(|dup\(')
    cyto_features['cyto_total_anomalies'] = (
        cyto_features['cyto_del_count'] + cyto_features['cyto_transloc_count'] + 
        cyto_features['cyto_inv_count'] + cyto_features['cyto_gain_count'] + 
        cyto_features['cyto_loss_count'] + cyto_features['cyto_other_count']
    )
    cyto_features['cyto_normal'] = cyto_col.str.match(r'^46,(xx|xy)(\[\d+\])?$', case=False).astype(int)
    cyto_features['cyto_complex'] = (
        (cyto_features['cyto_total_anomalies'] >= 3) | 
        cyto_col.str.contains('complex', case=False, na=False)
    ).astype(int)
    chromosomes = [str(i) for i in range(1, 23)] + ['X', 'Y']
    for chrom in chromosomes:
        pattern = rf'(\b|[,\(]){chrom}([,;:\)\[]|[pq])'
        cyto_features[f'cyto_chr{chrom}_affected'] = cyto_col.str.contains(
            pattern, case=False, na=False, regex=True
        ).astype(int)
    cyto_features['cyto_monosomy7'] = cyto_col.str.contains(r'-7[^0-9]|^45.*-7', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_trisomy8'] = cyto_col.str.contains(r'\+8[^0-9]|^47.*\+8', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del5q'] = cyto_col.str.contains(r'del\(5\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del20q'] = cyto_col.str.contains(r'del\(20\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr3_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(3[;,:\)]', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr7_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(7[;,:\)]', case=False, na=False, regex=True).astype(int)
    return cyto_features.fillna(0)

def create_molecular_features(molecular_df, patient_ids, top_n_genes=20):
    mol_features = pd.DataFrame({'ID': patient_ids})
    mutation_counts = molecular_df.groupby('ID').size().to_frame('mutation_count_total')
    mol_features = mol_features.merge(mutation_counts, on='ID', how='left')
    vaf_stats = molecular_df.groupby('ID')['VAF'].agg([
        ('vaf_mean', 'mean'), ('vaf_max', 'max'), ('vaf_sum', 'sum')
    ]).reset_index()
    mol_features = mol_features.merge(vaf_stats, on='ID', how='left')
    effect_counts = molecular_df.groupby(['ID', 'EFFECT']).size().unstack(fill_value=0)
    effect_counts.columns = [f'effect_{col}' for col in effect_counts.columns]
    mol_features = mol_features.merge(effect_counts.reset_index(), on='ID', how='left')
    top_genes_list = molecular_df['GENE'].value_counts().head(top_n_genes).index.tolist()
    for gene in top_genes_list:
        gene_mutations = molecular_df[molecular_df['GENE'] == gene].groupby('ID').size()
        mol_features[f'gene_{gene}_count'] = mol_features['ID'].map(gene_mutations)
        gene_present = molecular_df[molecular_df['GENE'] == gene]['ID'].unique()
        mol_features[f'gene_{gene}_present'] = mol_features['ID'].isin(gene_present).astype(int)
    feature_cols = [col for col in mol_features.columns if col != 'ID']
    mol_features[feature_cols] = mol_features[feature_cols].fillna(0)
    return mol_features.set_index('ID')

print("✓ Feature engineering functions defined")

In [ ]:
# Create features and prepare data (same as V4)
cyto_features_train = create_cytogenetic_features(clinical_train)
train_patient_ids = clinical_train['ID'].unique()
mol_features_train = create_molecular_features(molecular_train, train_patient_ids, top_n_genes=20)

target_clean = target_train.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
target_clean['OS_STATUS'] = target_clean['OS_STATUS'].astype(bool)
target_clean = target_clean.set_index('ID')

clinical_train_clean = clinical_train[clinical_train['ID'].isin(target_clean.index)].copy()
clinical_train_clean = clinical_train_clean.set_index('ID').loc[target_clean.index]

numeric_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
X_numeric = clinical_train_clean[numeric_features].copy()
center_encoded = pd.get_dummies(clinical_train_clean['CENTER'], prefix='CENTER', drop_first=True)
X_clinical = pd.concat([X_numeric, center_encoded], axis=1)

mol_features_train_aligned = mol_features_train.reindex(X_clinical.index, fill_value=0)
cyto_features_train_aligned = cyto_features_train.reindex(X_clinical.index, fill_value=0)
X_all = pd.concat([X_clinical, mol_features_train_aligned, cyto_features_train_aligned], axis=1)

# Select 90 features
available_features = [f for f in BEST_FEATURES if f in X_all.columns]
X_selected = X_all[available_features].copy()

# Imputation
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_selected),
    index=X_selected.index,
    columns=X_selected.columns
)

y_surv = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_clean)

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(
    X_imputed, y_surv, test_size=0.3, random_state=42,
    stratify=target_clean['OS_STATUS'].astype(int)
)

print(f"✓ Data prepared: {X_train.shape[0]} train, {X_val.shape[0]} val, {X_imputed.shape[1]} features")

## 4. Model 1: RandomSurvivalForest

In [ ]:
print("="*60)
print("MODEL 1: RANDOM SURVIVAL FOREST")
print("="*60)

start_time = time()

rsf = RandomSurvivalForest(
    n_estimators=350,
    min_samples_split=27,
    min_samples_leaf=5,
    max_features=0.2,
    max_depth=23,
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

rsf.fit(X_train, y_train)

y_pred_rsf_train = rsf.predict(X_train)
y_pred_rsf_val = rsf.predict(X_val)

c_index_rsf_train = concordance_index_censored(
    y_train['OS_STATUS'], y_train['OS_YEARS'], y_pred_rsf_train
)[0]

c_index_rsf_val = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_rsf_val
)[0]

elapsed = time() - start_time

print(f"\nTrain C-index: {c_index_rsf_train:.4f}")
print(f"Val C-index:   {c_index_rsf_val:.4f}")
print(f"Time: {elapsed:.1f}s")

## 5. Model 2: Gradient Boosting Survival Analysis

In [ ]:
print("="*60)
print("MODEL 2: GRADIENT BOOSTING SURVIVAL ANALYSIS")
print("="*60)

start_time = time()

gbm = GradientBoostingSurvivalAnalysis(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=30,
    min_samples_leaf=10,
    subsample=0.8,
    random_state=42,
    verbose=0
)

gbm.fit(X_train, y_train)

y_pred_gbm_train = gbm.predict(X_train)
y_pred_gbm_val = gbm.predict(X_val)

c_index_gbm_train = concordance_index_censored(
    y_train['OS_STATUS'], y_train['OS_YEARS'], y_pred_gbm_train
)[0]

c_index_gbm_val = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_gbm_val
)[0]

elapsed = time() - start_time

print(f"\nTrain C-index: {c_index_gbm_train:.4f}")
print(f"Val C-index:   {c_index_gbm_val:.4f}")
print(f"Time: {elapsed:.1f}s")

## 6. Model 3: Cox Proportional Hazards

In [ ]:
print("="*60)
print("MODEL 3: COX PROPORTIONAL HAZARDS")
print("="*60)

start_time = time()

cox = CoxPHSurvivalAnalysis(alpha=0.1, n_iter=1000)
cox.fit(X_train, y_train)

y_pred_cox_train = cox.predict(X_train)
y_pred_cox_val = cox.predict(X_val)

c_index_cox_train = concordance_index_censored(
    y_train['OS_STATUS'], y_train['OS_YEARS'], y_pred_cox_train
)[0]

c_index_cox_val = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_cox_val
)[0]

elapsed = time() - start_time

print(f"\nTrain C-index: {c_index_cox_train:.4f}")
print(f"Val C-index:   {c_index_cox_val:.4f}")
print(f"Time: {elapsed:.1f}s")

## 7. Ensemble: Weighted Average

In [ ]:
print("="*60)
print("ENSEMBLE MODEL (Weighted Average)")
print("="*60)

# Weighted average
y_pred_ensemble_train = (
    ENSEMBLE_WEIGHTS['rsf'] * y_pred_rsf_train + 
    ENSEMBLE_WEIGHTS['gbm'] * y_pred_gbm_train + 
    ENSEMBLE_WEIGHTS['cox'] * y_pred_cox_train
)

y_pred_ensemble_val = (
    ENSEMBLE_WEIGHTS['rsf'] * y_pred_rsf_val + 
    ENSEMBLE_WEIGHTS['gbm'] * y_pred_gbm_val + 
    ENSEMBLE_WEIGHTS['cox'] * y_pred_cox_val
)

c_index_ensemble_train = concordance_index_censored(
    y_train['OS_STATUS'], y_train['OS_YEARS'], y_pred_ensemble_train
)[0]

c_index_ensemble_val = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_ensemble_val
)[0]

print(f"\nTrain C-index: {c_index_ensemble_train:.4f}")
print(f"Val C-index:   {c_index_ensemble_val:.4f}")

## 8. Performance Comparison

In [ ]:
print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)

results = pd.DataFrame([
    {'Model': 'RSF', 'Train': c_index_rsf_train, 'Val': c_index_rsf_val, 'Overfitting': c_index_rsf_train - c_index_rsf_val},
    {'Model': 'GBM', 'Train': c_index_gbm_train, 'Val': c_index_gbm_val, 'Overfitting': c_index_gbm_train - c_index_gbm_val},
    {'Model': 'Cox', 'Train': c_index_cox_train, 'Val': c_index_cox_val, 'Overfitting': c_index_cox_train - c_index_cox_val},
    {'Model': 'Ensemble', 'Train': c_index_ensemble_train, 'Val': c_index_ensemble_val, 'Overfitting': c_index_ensemble_train - c_index_ensemble_val}
])

print(results.to_string(index=False))

# Best individual model
best_individual = results.iloc[:3]['Val'].max()
improvement = c_index_ensemble_val - best_individual

print(f"\n✓ Best individual model: {best_individual:.4f}")
print(f"✓ Ensemble: {c_index_ensemble_val:.4f}")
print(f"✓ Improvement: {improvement:+.4f}")

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

models = ['RSF', 'GBM', 'Cox', 'Ensemble']
scores = [c_index_rsf_val, c_index_gbm_val, c_index_cox_val, c_index_ensemble_val]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#27AE60']

bars = ax.bar(models, scores, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.axhline(y=0.74, color='green', linestyle='--', linewidth=2, alpha=0.5, label='V4 Baseline: 0.74')
ax.axhline(y=0.76, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Target: 0.76')

ax.set_ylabel('C-index (Validation)', fontsize=12)
ax.set_title('Model Performance - V5 Ensemble vs Individual Models', fontsize=14, fontweight='bold')
ax.set_ylim([0.72, max(0.77, max(scores) + 0.01)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Test Set Preparation & Final Predictions

In [ ]:
print("="*60)
print("TEST SET PREPARATION")
print("="*60)

# Create features for test (same as V4)
cyto_features_test = create_cytogenetic_features(clinical_test)
test_patient_ids = clinical_test['ID'].unique()
mol_features_test = create_molecular_features(molecular_test, test_patient_ids, top_n_genes=20)

clinical_test_indexed = clinical_test.set_index('ID')
X_numeric_test = clinical_test_indexed[numeric_features].copy()
center_encoded_test = pd.get_dummies(clinical_test_indexed['CENTER'], prefix='CENTER', drop_first=True)
for col in center_encoded.columns:
    if col not in center_encoded_test.columns:
        center_encoded_test[col] = 0
center_encoded_test = center_encoded_test[center_encoded.columns]
X_clinical_test = pd.concat([X_numeric_test, center_encoded_test], axis=1)

mol_features_test_aligned = mol_features_test.reindex(X_clinical_test.index, fill_value=0)
cyto_features_test_aligned = cyto_features_test.reindex(X_clinical_test.index, fill_value=0)
X_test_all = pd.concat([X_clinical_test, mol_features_test_aligned, cyto_features_test_aligned], axis=1)

# Add missing features
for feature in available_features:
    if feature not in X_test_all.columns:
        X_test_all[feature] = 0

X_test_selected = X_test_all[available_features].copy()
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test_selected),
    index=X_test_selected.index,
    columns=X_test_selected.columns
)

print(f"✓ Test set prepared: {X_test_imputed.shape}")

In [ ]:
print("\nRetraining all models on full training set...")

# Retrain RSF
rsf_final = RandomSurvivalForest(
    n_estimators=350, min_samples_split=27, min_samples_leaf=5,
    max_features=0.2, max_depth=23, bootstrap=True,
    n_jobs=-1, random_state=42, verbose=0
)
rsf_final.fit(X_imputed, y_surv)
print("✓ RSF trained")

# Retrain GBM
gbm_final = GradientBoostingSurvivalAnalysis(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    min_samples_split=30, min_samples_leaf=10, subsample=0.8,
    random_state=42, verbose=0
)
gbm_final.fit(X_imputed, y_surv)
print("✓ GBM trained")

# Retrain Cox
cox_final = CoxPHSurvivalAnalysis(alpha=0.1, n_iter=1000)
cox_final.fit(X_imputed, y_surv)
print("✓ Cox trained")

print("\n✓ All models retrained on full dataset")

In [ ]:
# Generate ensemble predictions
y_pred_rsf_test = rsf_final.predict(X_test_imputed)
y_pred_gbm_test = gbm_final.predict(X_test_imputed)
y_pred_cox_test = cox_final.predict(X_test_imputed)

# Ensemble
y_pred_ensemble_test = (
    ENSEMBLE_WEIGHTS['rsf'] * y_pred_rsf_test + 
    ENSEMBLE_WEIGHTS['gbm'] * y_pred_gbm_test + 
    ENSEMBLE_WEIGHTS['cox'] * y_pred_cox_test
)

# Convert to risk scores (normalized 0-1, high = high risk)
min_pred = y_pred_ensemble_test.min()
max_pred = y_pred_ensemble_test.max()
risk_scores = 1 - (y_pred_ensemble_test - min_pred) / (max_pred - min_pred)

submission = pd.DataFrame({
    'ID': X_test_imputed.index,
    'risk_score': risk_scores
})

submission_path = f"{DATA_PATH}\\submission_v5_ensemble.csv"
submission.to_csv(submission_path, index=False)

print("="*60)
print("SUBMISSION GENERATED")
print("="*60)
print(f"File: {submission_path}")
print(f"Predictions: {len(submission)}")
print(f"\n📊 Risk Scores (0-1, 1=high risk):")
print(submission['risk_score'].describe())

## 10. Summary

In [ ]:
print("="*60)
print("VERSION 5 - ENSEMBLE MODEL SUMMARY")
print("="*60)

print("\n📊 INDIVIDUAL MODELS:")
print(f"  RSF:   {c_index_rsf_val:.4f}")
print(f"  GBM:   {c_index_gbm_val:.4f}")
print(f"  Cox:   {c_index_cox_val:.4f}")

print("\n🔧 ENSEMBLE:")
print(f"  Weights: RSF={ENSEMBLE_WEIGHTS['rsf']}, GBM={ENSEMBLE_WEIGHTS['gbm']}, Cox={ENSEMBLE_WEIGHTS['cox']}")
print(f"  C-index: {c_index_ensemble_val:.4f}")

print("\n📈 IMPROVEMENT:")
print(f"  vs V4 (0.7404): {c_index_ensemble_val - 0.7404:+.4f}")
print(f"  vs Best individual: {improvement:+.4f}")

if c_index_ensemble_val >= 0.76:
    print("\n🏆 TARGET EXCEEDED! (C-index ≥ 0.76)")
elif c_index_ensemble_val >= 0.75:
    print("\n🎯 EXCELLENT! (C-index ≥ 0.75)")
elif c_index_ensemble_val > 0.74:
    print("\n✅ IMPROVED! Better than V4")

print("\n✅ OUTPUT:")
print(f"  Submission: submission_v5_ensemble.csv")
print(f"  Features: {X_imputed.shape[1]}")
print(f"  Models: 3 (RSF, GBM, Cox)")
print(f"  Ready for submission!")